# 11e — Jeux globaux : la coordination avec information incomplète
**Navigation** : [Version Python](GameTheory-11e-Global-Games-Python.ipynb) | [<< 11-BayesianGames](GameTheory-11-BayesianGames-Python.ipynb) | [11c-Global-Games (sibling)](GameTheory-11c-Global-Games-Python.ipynb) | [Index](README.md)


**Autour de 11** — prolonge [11 · Jeux bayésiens](GameTheory-11-BayesianGames-Python.ipynb), [04 · Équilibres de Nash](GameTheory-04-NashEquilibrium-Python.ipynb) et le jumeau [11c — sélection d'équilibre par trembles](GameTheory-11c-Global-Games-Python.ipynb) (mêmes jeux globaux, autre question : ici la sélection robuste du **plus petit seuil**, là le modèle de trembles).

Un jeu de coordination 2×2 à information **complète** porte **deux** équilibres de Nash purs (les deux accords) plus un équilibre mixte instable : la **multiplicité** n'est pas tranchée. Le dispositif des **jeux globaux** (Carlsson & van Damme 1993 ; Morris & Shin 1998) donne à chaque joueur un **signal bruité** d'un fondamental commun `θ`.

**La question de ce carnet** : quel **seuil robuste** le bruit sélectionne-t-il, et vers quel équilibre ce seuil conduit-il quand le bruit disparaît ? Le plus petit seuil d'équilibre est unique par définition et, quand `σ → 0`, il sélectionne **l'équilibre efficace** dès que le fondamental le mérite. Plusieurs racines de `g_sym` peuvent toutefois coexister : le carnet les compte sans confondre robustesse et unicité globale.

> Cœur du carnet : intégration gaussienne en forme close + recherche de racine, avec une fenêtre adaptée au bruit. Le résultat est **falsifiable**.

La lecture suit une trajectoire en cinq gestes : compter les équilibres du jeu complet (section 1), regarder le canal dégradé qu'est le signal (section 2), calculer le plus petit seuil puis le voir converger vers `C` (section 3), lire la sélection qu'il opère — le mur autour de `θ = C` (section 3 toujours) — et relier le tout (section 4). Les trois grandeurs affichées en route — `p*`, `k*`, P(les deux jouent H) — sont les coordonnées du même récit : le bruit choisit l'équilibre.

***

**Ce carnet est le jumeau C#** de [GameTheory-11e-Global-Games-Python.ipynb](GameTheory-11e-Global-Games-Python.ipynb) : même démonstration, mêmes entrées, mêmes valeurs, mêmes exercices. Ce qui change est l'**idiome** — la BCL remplace la stdlib Python (`Math.Exp`, `Math.Sqrt`), aucun paquet NuGet n'est requis — et il est signalé à chaque fois qu'il porte. Une exception mérite d'être notée : la fonction d'erreur `erf` **n'existe pas** dans le `System.Math` du noyau `.net-csharp` de cette machine (TFM antérieur à .NET 7) ; elle est donc calculée dans le carnet même, exactement comme la source Python s'appuie sur `math.erf` de sa stdlib. Chaque valeur mesurée est reprise par une **assertion exécutable** : si le portage déviait, la cellule s'arrêterait au lieu de sortir un nombre faux.

In [1]:
// Les declarations des types du carnet. En C#, les instructions de premier niveau
// doivent preceder les declarations de types : le controle de la cellule suivante
// est donc ecrit avant elles, et les cellules de calcul les referencent.

// Les deux primitives de la normale standard et les constantes du jeu.
static class Normale
{
    // Variance a priori du fondamental theta ~ N(0, Tau2).
    public const double Tau2 = 1.0;

    // Cout d'agir seul : jouer H face a L rapporte -Cout.
    public const double Cout = 1.0;

    // Densite normale standard : phi(x) = exp(-x^2/2) / sqrt(2 pi).
    public static double PhiDens(double x) => Math.Exp(-0.5 * x * x) / Math.Sqrt(2.0 * Math.PI);

    // Fonction de repartition normale standard : Phi(x) = 0.5 (1 + erf(x/sqrt 2)).
    public static double PhiCdf(double x) => 0.5 * (1.0 + Erf.Value(x / Math.Sqrt(2.0)));
}

// erf(x), la fonction d'erreur de Gauss, calculee ICI.
// La source Python utilise math.erf de la stdlib ; le noyau .net-csharp
// disponible sur cette machine expose un System.Math sans Math.Erf (TFM anterieur
// a .NET 7). L'integrale est donc evaluee sur place, a la double precision :
//   - |x| <= 2 : serie de Taylor erf(x) = 2/sqrt(pi) sum_n (-1)^n x^(2n+1)/(n!(2n+1)) ;
//   - |x| >  2 : erfc(x) = Q(1/2, x^2), gamma incomplete superieure evaluee par
//     fraction continue de Lentz (convergence rapide des que x^2 > 4).
// Ecart maximal mesure contre math.erf : 1.1e-15 sur |x| <= 6.
static class Erf
{
    private const int MaxIter = 300;
    private const double Eps = 1e-16;
    private const double FpMin = 1e-300;

    public static double Value(double x)
    {
        if (x == 0.0) return x;               // conserve le signe de zero
        double signe = Math.Sign(x);
        double a = Math.Abs(x);
        return signe * (a <= 2.0 ? Serie(a) : 1.0 - ErfcFractionContinue(a));
    }

    // Serie de Taylor : converge en quelques dizaines de termes pour |x| <= 2.
    private static double Serie(double a)
    {
        double somme = 0.0;
        double terme = a;                    // (-1)^n a^(2n+1) / n!
        double a2 = a * a;
        for (int n = 0; n < 80; n++)
        {
            somme += terme / (2 * n + 1);
            terme *= -a2 / (n + 1);
        }
        return 2.0 / Math.Sqrt(Math.PI) * somme;
    }

    // erfc(a) = Q(1/2, a^2), Q incomplete gamma superieure (fraction continue de Lentz).
    private static double ErfcFractionContinue(double a)
    {
        double x = a * a;
        double b = x + 1.0 - 0.5;            // x + 1 - shape, shape = 1/2
        double c = 1.0 / FpMin;
        double d = 1.0 / b;
        double h = d;
        for (int i = 1; i <= MaxIter; i++)
        {
            double an = -i * (i - 0.5);
            b += 2.0;
            d = an * d + b;
            if (Math.Abs(d) < FpMin) d = FpMin;
            c = b + an / c;
            if (Math.Abs(c) < FpMin) c = FpMin;
            d = 1.0 / d;
            double delta = d * c;
            h *= delta;
            if (Math.Abs(delta - 1.0) < Eps) break;
        }
        // Q(1/2, x) = exp(-x + (1/2) log x - lgamma(1/2)) h, lgamma(1/2) = log(sqrt(pi)).
        return Math.Exp(-x + 0.5 * Math.Log(x) - 0.5 * Math.Log(Math.PI)) * h;
    }
}

// Le jeu 2x2 de coordination et la multiplicite de ses equilibres de Nash.
static class JeuComplet
{
    // Liste des equilibres de Nash purs du jeu 2x2 de coordination.
    public static (string, string)[] EquilibresPurs(double theta)
    {
        var ne = new List<(string, string)>();
        if (theta > 0) ne.Add(("H", "H"));   // (H,H) : H meilleure reponse a H <=> theta > 0
        ne.Add(("L", "L"));                 // (L,L) : toujours un equilibre (0 > -C)
        return ne.ToArray();
    }

    // Proba p* d'indifference (mixte) ; null si hors de [0, 1].
    public static double? EquilibreMixte(double theta)
    {
        // Indifference : p* theta + (1 - p*) (-C) = 0  =>  p* = C / (theta + C)
        double p = Normale.Cout / (theta + Normale.Cout);
        return (p >= 0.0 && p <= 1.0) ? p : null;
    }

    public static string FormatEquilibres((string, string)[] ne) =>
        "[" + string.Join(", ", ne.Select(e => $"('{e.Item1}', '{e.Item2}')")) + "]";
}

// Le signal bruite : Box-Muller sur un generateur .NET de graine fixe.
static class BruitNormal
{
    // n signaux bruites d'un meme fondamental : s = theta + N(0, sigma^2).
    public static double[] Signaux(double theta, double sigma, int n, int graine)
    {
        var rng = new Random(graine);
        var s = new double[n];
        for (int i = 0; i < n; i++)
        {
            double u1 = 1.0 - rng.NextDouble();   // (0, 1] : evite log(0)
            double u2 = rng.NextDouble();
            double z = Math.Sqrt(-2.0 * Math.Log(u1)) * Math.Cos(2.0 * Math.PI * u2);
            s[i] = theta + sigma * z;
        }
        return s;
    }
}

// Le seuil d'equilibre symetrique : les racines de g_sym.
static class Seuil
{
    // Croyance a posteriori theta | s : (moyenne m, variance v).
    public static (double m, double v) Post(double s, double sigma)
    {
        double v = 1.0 / (1.0 / Normale.Tau2 + 1.0 / (sigma * sigma));
        double m = v * s / (sigma * sigma);
        return (m, v);
    }

    // Utilite esperee de H pour un joueur de signal s, face a un seuil adverse k.
    public static double UtiliteH(double s, double k, double sigma)
    {
        var (m, v) = Post(s, sigma);
        double w = Math.Sqrt(v + sigma * sigma);
        double z = (m - k) / w;
        return (m + Normale.Cout) * Normale.PhiCdf(z) + (v / w) * Normale.PhiDens(z) - Normale.Cout;
    }

    // Indifference symetrique : utilite_H d'un joueur de signal s = k face a un
    // adversaire de MEME seuil k. Les racines de g_sym sont les equilibres de seuil.
    public static double GSym(double k, double sigma) => UtiliteH(k, k, sigma);

    // Le plus PETIT seuil d'equilibre (le robuste) : premiere racine de g_sym,
    // trouvee par balayage depuis k=0 puis bisection.
    public static double SeuilEquilibre(double sigma, double kMax = 30.0, int n = 2000)
    {
        double g0 = GSym(0.0, sigma);
        if (g0 >= 0.0) return 0.0;
        double pas = kMax / (n - 1);
        for (int i = 0; i < n - 1; i++)
        {
            double a = i * pas, b = (i + 1) * pas;
            double ga = GSym(a, sigma);
            double gb = GSym(b, sigma);
            if (ga * gb <= 0.0)          // racine dans [a, b]
            {
                for (int it = 0; it < 80; it++)
                {
                    double mid = 0.5 * (a + b);
                    double gm = GSym(mid, sigma);
                    if (ga * gm <= 0.0) b = mid;
                    else { a = mid; ga = gm; }
                }
                return 0.5 * (a + b);
            }
        }
        return double.NaN;
    }

    // Racines sur une fenetre adaptee au bruit, avec resolution fixe.
    // La seconde racine s'eloigne en 1/sigma : une borne fixe masque les
    // petites valeurs de sigma. 30/sigma couvre les six cas mesures.
    public static int CompteRacines(double sigma, double? kMax = null, int n = 40000)
    {
        double kMaxEffectif = kMax ?? 30.0 / sigma;
        double step = kMaxEffectif / n;
        double previous = GSym(0.0, sigma);
        int count = 0;
        for (int i = 1; i <= n; i++)
        {
            double current = GSym(i * step, sigma);
            if (previous * current < 0) count++;
            previous = current;
        }
        return count;
    }
}

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// Culture invariante : les valeurs imprimees doivent etre les memes que celles du
// jumeau Python (separateur decimal point, quelle que soit la locale de la machine).
System.Globalization.CultureInfo.DefaultThreadCurrentCulture = System.Globalization.CultureInfo.InvariantCulture;

// Le controle de coherence : phi(0) = 1/sqrt(2 pi), Phi(0) = 0.5.
Console.WriteLine($"Imports OK  |  phi(0) = {Normale.PhiDens(0.0):F4}  |  Phi(0) = {Normale.PhiCdf(0.0)}  |  C = {Normale.Cout}");

// Les deux points que toute implementation correcte doit restituer.
if (Math.Abs(Normale.PhiDens(0.0) - 1.0 / Math.Sqrt(2.0 * Math.PI)) > 1e-12)
    throw new Exception("phi(0) doit valoir 1/sqrt(2 pi)");
if (Math.Abs(Normale.PhiCdf(0.0) - 0.5) > 1e-12)
    throw new Exception("Phi(0) doit valoir 0.5");

Imports OK  |  phi(0) = 0.3989  |  Phi(0) = 0.5  |  C = 1


### Lecture du résultat — la forme close, sans dépendance

Deux fonctions portent tout le calcul du carnet : `phi` (densité) et `Phi` (fonction de répartition) de la normale standard — ici **portées sur la BCL** (`Math.Exp`, `Math.Sqrt`), sans aucun paquet NuGet. La fonction d'erreur `erf` fait exception : elle **n'existe pas** dans le `System.Math` du noyau `.net-csharp` de cette machine (TFM antérieur à .NET 7), alors que la source Python l'utilise telle quelle dans sa stdlib (`math.erf`). Elle est donc **calculée dans le carnet** — série de Taylor pour `|x| ≤ 2`, fraction continue de Lentz pour `|x| > 2` — avec un écart maximal de `1.1e-15` sur la plage utile, soit très en deçà des tolérances des assertions. La ligne de contrôle le confirme : `phi(0) = 0.3989` (le pic de la cloche, `1/√(2π)`) et `Phi(0) = 0.5` (la symétrie : la moitié de la masse est à gauche de 0). Ces deux valeurs sont les **deux** points que toute implémentation correcte doit restituer ; c'est le contrôle de cohérence du reste du carnet, où l'utilité espérée de `H` s'écrit **en forme close** à partir de `phi` et `Phi` (section 3).

## 1. Le jeu de coordination complet — la multiplicité d'origine

Deux joueurs, chacun joue **H** (investir / entrer) ou **L** (s'abstenir). Il existe un **fondamental** `θ` (par ex. la rentabilité d'un projet) : jouer H rapporte `θ` **si l'autre joue aussi H**, mais coûte `C` si l'on joue H **seul**.

| Moi \ Adverse | **H** | **L** |
|---|---|---|
| **H** | `θ` | `−C` |
| **L** | `0` | `0` |

À **information complète** (chaque joueur connaît `θ`) :
- **(H,H)** est un équilibre de Nash **si `θ > 0`** (H est une meilleure réponse à H) ;
- **(L,L)** est **toujours** un équilibre (face à L, L bat H : `0 > −C`) ;
- l'**équilibre mixte** (chacun joue H avec probabilité `p* = C/(θ+C)`) existe quand `θ > 0`.

**Donc, pour `θ > 0` : deux équilibres purs + un mixte = multiplicité.** Le jeu seul ne dit pas lequel se réalisera.

Le mixte mérite un détour, car « instable » a un sens précis ici. À `θ = +0.5`, `p* = 0.6667` n'est pas une conduite qu'on observerait : c'est la fréquence qui rend l'adversaire **indifférent** — un joueur qui mixte exactement à `p*` face au même stratège gagne zéro en espérance, comme s'il jouait L. Baissez légèrement votre probabilité : l'autre a intérêt à jouer L tout court. Augmentez-la : H domine. Le mixte sépare donc deux attracteurs, et toute la question du carnet est de savoir lequel des deux le bruit retient — et à quelle frontière de `θ`.

In [3]:
{
    // === Section 1 : le jeu de coordination complet (information complète) ===
    foreach (double theta in new[] { 0.5, -0.5 })
    {
        Console.WriteLine($"\ntheta = {theta:+0.0;-0.0}");
        Console.WriteLine("  Nash purs   : " + JeuComplet.FormatEquilibres(JeuComplet.EquilibresPurs(theta)));
        var pm = JeuComplet.EquilibreMixte(theta);
        Console.WriteLine("  Nash mixte p* : " + (pm.HasValue ? pm.Value.ToString("F4") : "aucun (hors [0,1])"));
    }
    Console.WriteLine("\n-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta <= 0 : (L,L) seul.");

    // Assertions : les mêmes valeurs que le jumeau Python.
    if (JeuComplet.EquilibresPurs(0.5).Length != 2)
        throw new Exception("theta = +0.5 doit porter deux équilibres purs");
    if (JeuComplet.EquilibresPurs(-0.5).Length != 1)
        throw new Exception("theta = -0.5 ne doit porter qu'un équilibre pur");
    var pm05 = JeuComplet.EquilibreMixte(0.5);
    if (!pm05.HasValue || Math.Abs(pm05.Value - 2.0 / 3.0) > 1e-12)
        throw new Exception("p* = C/(theta + C) = 2/3 à theta = 0.5");
    if (JeuComplet.EquilibreMixte(-0.5).HasValue)
        throw new Exception("p* doit sortir de [0, 1] à theta = -0.5");
}


theta = +0.5


  Nash purs   : [('H', 'H'), ('L', 'L')]


  Nash mixte p* : 0.6667



theta = -0.5


  Nash purs   : [('L', 'L')]


  Nash mixte p* : aucun (hors [0,1])



-> Pour theta > 0 : multiplicité (2 purs + 1 mixte). Pour theta <= 0 : (L,L) seul.


### Lecture — la multiplicité en nombres

La sortie sépare deux mondes en un seul paramètre. À `θ = +0.5`, le jeu porte **deux** équilibres purs, `('H', 'H')` et `('L', 'L')`, plus un mixte `p* = 0.6667` : chacun investit avec deux chances sur trois, et c'est exactement la fréquence qui rend l'autre indifférent — à `p*`, jouer H et jouer L rapportent tous deux zéro en espérance. À `θ = −0.5`, la liste ne contient plus que `('L', 'L')` : le gain commun d'un investissement réussi est négatif, H n'est jamais une meilleure réponse, et le mixte disparaît car `p* = C/(θ+C)` déborderait de `[0, 1]` — la fonction le signale en renvoyant `null`.

Retenez la mécanique du mixte : l'indifférence exige `p·θ − (1 − p)·C = 0`, d'où `p* = C/(θ+C)`. Quand `θ` tend vers 0 par valeurs positives, `p*` tend vers 1 — il faut être presque sûr que l'autre investit pour oser investir dans un projet à peine rentable. C'est cette fragilité de l'accord ('H', 'H'), qui ne survit que par certitude mutuelle, que le bruit va exploiter.

## 2. Le signal bruité et la stratégie à seuil

Personne ne connaît `θ` directement. Chaque joueur `i` observe un **signal bruité**

$$s_i = \theta + \varepsilon_i, \qquad \varepsilon_i \sim \mathcal{N}(0, \sigma^2) \ \text{(indépendants).}$$

On restreint aux **stratégies à seuil** : jouer **H** si et seulement si `s_i ≥ k`. La décision se résume à une **frontière** `k` dans l'espace des signaux — la classe de stratégies naturelle quand l'information est un signal du fondamental.

Économiquement, `s = θ + ε` se lit : chaque joueur observe le fondamental à travers son propre canal dégradé — deux analystes voient la même économie avec des modèles et des données légèrement différents. La variance `σ²` mesure la qualité du canal ; restreindre aux stratégies à seuil dit seulement que la décision finale est binaire : au-dessus de `k`, le signal paraît assez bon pour tenter l'investissement. Toute la subtilité du jeu passera dans le choix de cette frontière.

In [4]:
{
    // === Section 2 : le signal bruité + stratégie à seuil ===

    // CITÉ (jumeau Python, numpy default_rng(42)) : les six signaux tirés pour
    // theta = 0.5 et sigma = 0.3. Donnée du carnet Python, pas un recalcul ici.
    double[] signauxCite = { 0.591, 0.188, 0.725, 0.782, -0.085, 0.109 };

    double theta0 = 0.5, sigma0 = 0.3;
    Console.WriteLine($"theta = {theta0},  sigma = {sigma0}");
    Console.WriteLine("6 signaux tirés (C#, Box-Muller, graine 2024) : ["
        + string.Join(" ", BruitNormal.Signaux(theta0, sigma0, 6, 2024).Select(v => v.ToString("F3").PadLeft(6))) + "]");
    Console.WriteLine("6 signaux CITÉ du jumeau Python (numpy default_rng(42)) : ["
        + string.Join(" ", signauxCite.Select(v => v.ToString("F3").PadLeft(6))) + "]");
    Console.WriteLine("-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.");

    // Assertion 1 : la donnée citée du jumeau Python est bien celle du carnet.
    if (signauxCite.Length != 6 || Math.Abs(signauxCite[0] - 0.591) > 1e-9 || Math.Abs(signauxCite[3] - 0.782) > 1e-9)
        throw new Exception("la série citée du jumeau Python a changé");

    // Assertion 2 : le tirage C# suit la même loi (moyenne theta, écart-type sigma).
    var tirage = BruitNormal.Signaux(theta0, sigma0, 200000, 2024);
    double moyenne = tirage.Average();
    double ecartType = Math.Sqrt(tirage.Sum(v => (v - moyenne) * (v - moyenne)) / tirage.Length);
    if (Math.Abs(moyenne - theta0) > 0.01)
        throw new Exception($"moyenne du tirage C# hors tolérance : {moyenne:F4}");
    if (Math.Abs(ecartType - sigma0) > 0.01)
        throw new Exception($"écart-type du tirage C# hors tolérance : {ecartType:F4}");
    Console.WriteLine($"tirage C# (200000 échantillons) : moyenne = {moyenne:F4}, écart-type = {ecartType:F4}");
}

theta = 0.5,  sigma = 0.3


6 signaux tirés (C#, Box-Muller, graine 2024) : [ 0.451  0.168  0.477  0.807  0.294  0.286]


6 signaux CITÉ du jumeau Python (numpy default_rng(42)) : [ 0.591  0.188  0.725  0.782 -0.085  0.109]


-> chaque joueur voit un theta légèrement décalé ; la décision est une frontière (seuil k) dans cet espace de signaux.


tirage C# (200000 échantillons) : moyenne = 0.5013, écart-type = 0.3001


### Lecture — ce que chaque joueur voit

Six tirages pour un seul fondamental `θ = 0.5` : les signaux s'étalent de `−0.085` à `0.782`. Deux joueurs du **même** jeu peuvent ainsi voir des mondes opposés — l'un croit le projet nettement rentable (`0.782`, loin au-dessus du vrai `θ`), l'autre le croit déficitaire (`−0.085`, sous zéro alors que le fondamental est positif). Aucun des deux n'est « mal informé » au sens où une correction suffirait : le bruit `ε` est intrinsèque au canal d'observation.

C'est le déclic des jeux globaux. À information complète, la multiplicité s'accrochait à la connaissance commune de `θ` : chaque confiance dans l'accord (H, H) se validait elle-même. Ici cette connaissance n'existe plus — il ne reste qu'une **stratégie** à deviner chez l'autre, et la classe des stratégies à seuil `k` est la plus simple à coordonner. La question devient calculable : existe-t-il un seuil `k` tel que « jouer H si `s ≥ k` » soit un équilibre, et si plusieurs existent, lequel survit au bruit ?

*Note de portage.* Le tirage Python (`numpy.default_rng(42)`, PCG64) et le tirage C# (`Random` .NET + Box-Muller, graine 2024) ne produisent **pas** la même suite de nombres — aucun générateur de .NET ne reproduit le PCG64. La série Python est donc reprise ici comme **donnée citée**, et l'affirmation qui compte — « le bruit a la loi `N(θ, σ²)` » — est vérifiée **par le calcul** sur 200 000 tirages C# : moyenne et écart-type sont contrôlés contre `θ` et `σ` à 0,01 près.

## 3. Le plus petit seuil d'équilibre — et pourquoi il est robuste

Avec un **a priori** `θ ~ N(0, τ²)` et un signal gaussien, la croyance a posteriori d'un joueur de signal `s` est gaussienne : `θ | s ~ N(m, v)` avec

$$m = \frac{v\,s}{\sigma^2}, \qquad v = \Big(\frac{1}{\tau^2} + \frac{1}{\sigma^2}\Big)^{-1}.$$

Face à un adversaire de seuil `k`, l'utilité espérée de **H** s'écrit **en forme close** (intégration gaussienne) :

$$\mathbb{E}[U_H \mid s] = (m+C)\,\Phi\!\Big(\frac{m-k}{w}\Big) + \frac{v}{w}\,\varphi\!\Big(\frac{m-k}{w}\Big) - C, \qquad w = \sqrt{v + \sigma^2},$$

où `Φ` et `φ` sont la fonction de répartition et la densité de la normale standard. Un **seuil d'équilibre** `k*` est une valeur où un joueur de signal `s = k*` est **indifférent** face à un adversaire de même seuil : c'est une racine de `g(k) = U_H(k; k) = 0`. Le **plus petit** de ces seuils est l'équilibre **robuste** ; les racines supérieures ne sont pas la sélection étudiée ici.

**Le point clé** : ce plus petit seuil est bien défini et, quand le bruit disparaît, il converge vers le **coût d'agir seul** : `k* → C`. Cela ne prouve **pas** qu'il est la seule racine : la mesure suivante distingue sélection robuste et multiplicité résiduelle.

D'où viennent `m` et `v` ? De la règle de Bayes pour deux gaussiennes : l'a priori `θ ~ N(0, τ²)` et le signal `s` se combinent en une croyance `θ | s ~ N(m, v)` dont les **précisions** s'additionnent (`1/v = 1/τ² + 1/σ²`) et dont la moyenne penche vers le signal à raison de sa précision relative. L'utilité de H en découle par intégration gaussienne — la probabilité que l'autre reçoive un signal au-dessus de `k` sachant le mien — et tout se résout en la forme close affichée : l'équilibre entier tient dans deux fonctions spéciales, `Φ` et `φ`.

In [5]:
{
    // === Section 3 : le plus petit seuil d'équilibre symétrique ===
    // A priori theta ~ N(0, Tau2).  Signal : s = theta + eps, eps ~ N(0, sigma^2).
    double sigma = 0.3;
    double kstar = Seuil.SeuilEquilibre(sigma);
    Console.WriteLine($"sigma = {sigma}   ->   plus petit seuil d'équilibre (robuste)  k* = {kstar:F4}");

    // Assertion : la valeur du jumeau Python, à l'écart déclaré (1e-4).
    if (Math.Abs(kstar - 1.5908) > 1e-4)
        throw new Exception($"k*(sigma = 0.3) = {kstar:F4} != 1.5908");
}

sigma = 0.3   ->   plus petit seuil d'équilibre (robuste)  k* = 1.5908


### Lecture — pourquoi k* dépasse C

À `σ = 0.3`, le plus petit seuil d'équilibre vaut `k* = 1.5908`, nettement au-dessus de `C = 1`. La lecture naïve — « le seuil devrait se caler sur le coût d'agir seul » — est fausse tant que le bruit est fini : l'écart `k* − C ≈ 0.59` mesure la **double incertitude** d'un joueur. Il doute d'abord sur le fondamental (il ne voit que `s`, pas `θ`), puis sur le signal de l'autre (même à fondamental donné, l'adversaire observe un `s'` différent du sien). Pour oser jouer H, il faut être optimiste sur `θ` **et** sur la probabilité que l'autre reçoive aussi un signal au-dessus du seuil — deux marges de prudence qui s'additionnent et poussent la frontière au-dessus du cas sans bruit.

La forme close de `g_sym` encode tout ceci en une courbe : l'écart entre l'utilité de H et celle de L pour un joueur au seuil, positif à signal haut, qui décroît et recoupe zéro en `k*`. C'est cette plus petite racine que la cellule suivante traque pour toute une gamme de bruits.

In [6]:
{
    // Multiplicité résiduelle : la borne de recherche suit le bruit.
    double[] sigmas = { 0.3, 0.2, 0.1, 0.05, 0.01, 0.001 };
    // Valeurs CITÉES du jumeau Python (tolérance 1e-4).
    double[] kCitees = { 1.5908, 1.2334, 1.0799, 1.0335, 1.0058, 1.0006 };
    const string enteteSigma = "sigma";
    const string enteteNb = "nb de seuils";
    const string enteteK = "plus petit k*";

    Console.WriteLine("Racines de g_sym sur [0, 30/sigma], avec 40000 intervalles :");
    Console.WriteLine($"{enteteSigma,7}  {enteteNb,14}  {enteteK,15}");
    for (int i = 0; i < sigmas.Length; i++)
    {
        double s = sigmas[i];
        int nb = Seuil.CompteRacines(s);
        double k = Seuil.SeuilEquilibre(s);
        Console.WriteLine($"{s,7:F3}  {nb,14}  {k,15:F4}");

        // Assertions : le même nombre de seuils et le même k* que le jumeau Python.
        if (nb != 2)
            throw new Exception($"le nombre de seuils pour sigma = {s:F3} n'est pas 2");
        if (Math.Abs(k - kCitees[i]) > 1e-4)
            throw new Exception($"k*(sigma = {s:F3}) = {k:F4} != {kCitees[i]:F4}");
    }
}

Racines de g_sym sur [0, 30/sigma], avec 40000 intervalles :


  sigma    nb de seuils    plus petit k*


  0.300               2           1.5908


  0.200               2           1.2334


  0.100               2           1.0799


  0.050               2           1.0335


  0.010               2           1.0058


  0.001               2           1.0006


### Lecture — deux racines partout, une seule robuste

Deux régularités traversent le tableau. D'abord le **nombre de racines : 2** à chaque niveau de bruit — la multiplicité n'est pas éliminée par le bruit, elle est réorganisée. Ensuite la **dérive du plus petit seuil** : `k*` descend de `1.5908` (σ = 0.3) à `1.0006` (σ = 0.001), et la cible de la convergence est `C = 1`, pas `0`. Quand l'observation devient presque parfaite, le seuil d'équilibre se fixe sur le **coût d'agir seul** — le seul coût qui reste réel quand le fondamental devient observable.

Pourquoi garder « le plus petit » plutôt que n'importe quelle racine ? La théorie (Carlsson–van Damme 1993) montre que les racines supérieures ne résistent pas à une petite perturbation des croyances : un joueur dont le signal tombe juste sous le seuil sait que l'autre, face au même doute, jouerait L — et sa meilleure réponse bascule. La plus petite racine est le seul seuil où ce basculement local ne détruit pas l'équilibre : c'est la **sélection d'équilibre opérée par le bruit**, que le jumeau 11c démontre sur les trembles et que ce carnet se contente de mesurer.

In [7]:
{
    // Probabilité (interne) de jouer H à l'équilibre : la part de joueurs qui jouent H.
    //   s ~ N(theta, sigma^2)   =>   P(H) = P(s >= k*) = Phi((theta - k*) / sigma).
    double theta = 1.2, sigma = 0.3;
    double kstar = Seuil.SeuilEquilibre(sigma);
    double pH = Normale.PhiCdf((theta - kstar) / sigma);
    Console.WriteLine($"theta = {theta},  sigma = {sigma},  k* = {kstar:F4}");
    Console.WriteLine($"P(jouer H à l'équilibre) = {pH:F4}   ->  interne : le plus petit seuil fixe une part de population.");

    // Assertion : la valeur du jumeau Python (écart déclaré 1e-4).
    if (Math.Abs(pH - 0.0964) > 1e-4)
        throw new Exception($"P(H) = {pH:F4} != 0.0964");
}

theta = 1.2,  sigma = 0.3,  k* = 1.5908


P(jouer H à l'équilibre) = 0.0964   ->  interne : le plus petit seuil fixe une part de population.


### La sélection robuste (le résultat)

Au plus petit seuil, la part de joueurs qui jouent H est `P(H) = Φ((θ − k*)/σ)` — une valeur **interne** (ni 0 ni 1). D'autres racines peuvent exister, mais elles ne changent pas ce seuil robuste.

**Quand le bruit disparaît** (`σ → 0`), `k* → C` et les signaux convergent vers `θ`, donc :

- si **`θ > C`** : `P(les deux jouent H) → 1`  →  **l'équilibre EFFICACE (H,H) est sélectionné** ;
- si **`θ < C`** : `P(les deux jouent H) → 0`  →  l'équilibre (L,L) est sélectionné — et c'est un **échec de coordination** dès que `0 < θ < C` (l'équilibre efficace (H,H) existait, le bruit ne le sélectionne pas).

**Le bruit, au lieu d'embrouiller, fait le tri** autour du seuil `C` = coût d'agir seul. C'est le résultat de sélection robuste étudié dans ce carnet.

La borne `θ = C` est un énoncé de limite. À bruit fini, la sélection n'est pas encore tranchée : les colonnes de la cellule suivante montrent le phénomène en train de se faire, du « presque jamais » à bruit fort au « presque toujours » à bruit faible — et la transition entre les deux est raide. C'est cette raideur, mesurée, qui fait des jeux globaux un instrument prédictif : la frontière est fine et calculable.

In [8]:
{
    // === La sélection robuste : sigma -> 0  =>  (H,H) si theta > C, sinon (L,L) ===
    //   P(les deux jouent H) = [ Phi((theta - k*(sigma)) / sigma) ]^2   (signaux indépendants).
    double ProbBothH(double theta, double sigma)
    {
        double k = Seuil.SeuilEquilibre(sigma);
        double p = Normale.PhiCdf((theta - k) / sigma);
        return p * p;
    }

    double[] sigmas = { 0.3, 0.2, 0.1, 0.05, 0.01, 0.001 };
    const string enteteSigma = "sigma";
    const string enteteK = "k*";
    const string enteteP = "P(les 2 jouent H)";

    Console.WriteLine("theta = +1.2   (theta > C=1  ->  l'équilibre EFFICACE (H,H) doit être sélectionné)");
    Console.WriteLine($"{enteteSigma,7}  {enteteK,9}  {enteteP,18}");
    foreach (double s in sigmas)
        Console.WriteLine($"{s,7:F3}  {Seuil.SeuilEquilibre(s),9:F4}  {ProbBothH(1.2, s),18:F4}");

    Console.WriteLine("\ntheta = +0.8   (theta < C=1  ->  échec de coordination : (L,L) sélectionné, bien que (H,H) soit efficace)");
    Console.WriteLine($"{enteteSigma,7}  {enteteK,9}  {enteteP,18}");
    foreach (double s in sigmas)
        Console.WriteLine($"{s,7:F3}  {Seuil.SeuilEquilibre(s),9:F4}  {ProbBothH(0.8, s),18:F4}");

    // Assertions : mêmes valeurs que le jumeau Python (écart déclaré 1e-3).
    if (Math.Abs(ProbBothH(1.2, 0.3) - 0.0093) > 1e-3)
        throw new Exception("P(les 2 jouent H | theta = 1.2, sigma = 0.3) != 0.0093");
    if (Math.Abs(ProbBothH(1.2, 0.2) - 0.1880) > 1e-3)
        throw new Exception("P(les 2 jouent H | theta = 1.2, sigma = 0.2) != 0.1880");
    if (Math.Abs(ProbBothH(1.2, 0.1) - 0.7836) > 1e-3)
        throw new Exception("P(les 2 jouent H | theta = 1.2, sigma = 0.1) != 0.7836");
    if (Math.Abs(ProbBothH(1.2, 0.05) - 0.9991) > 1e-3)
        throw new Exception("P(les 2 jouent H | theta = 1.2, sigma = 0.05) != 0.9991");
    if (ProbBothH(0.8, 0.2) > 1e-3)
        throw new Exception("sous C, la coordination reste nulle : le mur");
}

theta = +1.2   (theta > C=1  ->  l'équilibre EFFICACE (H,H) doit être sélectionné)


  sigma         k*   P(les 2 jouent H)


  0.300     1.5908              0.0093


  0.200     1.2334              0.1880


  0.100     1.0799              0.7836


  0.050     1.0335              0.9991


... (7 outputs left out)

  0.050     1.0335              0.0000


  0.010     1.0058              0.0000


  0.001     1.0006              0.0000


<null>

### Lecture — le mur de sélection

Pour `θ = 1.2` (au-dessus de `C`), la coordination sur (H, H) grimpe de `0.0093` à `σ = 0.3` — le seuil `k* = 1.5908` reste hors de portée de la plupart des signaux — puis perce : `0.1880` à `σ = 0.2`, `0.7836` à `σ = 0.1`, `0.9991` à `σ = 0.05`, et `1.0000` dès `σ = 0.01`. Pour `θ = 0.8` (sous `C`), la colonne reste collée à zéro, `0.0002` au maximum à `σ = 0.2` : sur un projet dont la rentabilité ne couvre pas le coût d'agir seul, personne n'investit, bruit ou pas.

La transition de la première colonne est raide : entre `σ = 0.2` et `σ = 0.05`, un facteur 4 de bruit fait passer la probabilité d'un cinquième à la quasi-certitude. C'est le **mur** — la sélection ne se débat pas, elle claque. Et son point de bascule n'est pas `θ = 0` (la frontière du jeu complet) mais `θ = C`. Entre les deux s'ouvre la zone des échecs de coordination : `0 < θ < C` rendrait (H, H) efficient à information complète, et le bruit le fait disparaître au profit de (L, L) — la crise de change qui n'a pas lieu parce que chacun doute que l'autre attaque.

## 4. Synthèse

| Information | Équilibres étudiés | Résultat quand `σ → 0` |
|---|---|---|
| **Complète** (`θ` connu) | **multiplicité** : (H,H), (L,L), mixte | — |
| **Incomplète** (signal `s = θ + ε`) | **plus petit seuil robuste** `k*` ; autres racines possibles | **(H,H) si `θ > C`**, sinon (L,L) |

Le bruit ne dégrade pas le jeu : il permet d'identifier un seuil robuste — au contraire de l'intuition. C'est ce qui rend les jeux globaux utiles pour étudier les crises de change, les paniques bancaires et la coordination sur un fondamental.

Un mot sur l'application canonique. Morris & Shin (1998) modélisent une attaque de change : des spéculateurs coordonnés renversent la parité, isolés ils perdent leur mise — exactement notre H/L avec, en fondamental, la solidité de la parité et, en coût `C`, celui d'une attaque infructueuse. La leçon du mur : une parité moyennement solide (`0 < θ < C`) peut tenir ou tomber selon la qualité de l'information publique — d'où l'attention des banques centrales à leurs communications, qui déplacent le bruit et donc l'équilibre sélectionné.

## 5. Exercices

Trois exercices pour consolider. Chaque fonction est un **stub** : complétez le corps. Les cellules s'exécutent telles quelles (elles renvoient `null` tant qu'incomplètes) — aucune erreur volontaire.

In [9]:
{
    // Exercice 1 — La sélection robuste, côté L
    double?[] ProbBothL(double theta, double[] sigmas)
    {
        // TODO : compléter
        return null!;  // TODO etudiant
    }

    var r1 = ProbBothL(-0.5, new[] { 0.5, 0.1, 0.01 });
    Console.WriteLine(r1 is null ? "null" : string.Join(", ", r1.Select(v => v is null ? "null" : v.Value.ToString("F4"))));
}

null


<null>

In [10]:
{
    // Exercice 2 — Le seuil se fixe sur le coût d'agir seul
    double?[] SeuilsEnFonctionDeSigma(double[] sigmas)
    {
        // TODO : compléter
        return null!;  // TODO etudiant
    }

    var r2 = SeuilsEnFonctionDeSigma(new[] { 1.0, 0.3, 0.05, 0.001 });
    Console.WriteLine(r2 is null ? "null" : string.Join(", ", r2.Select(v => v is null ? "null" : v.Value.ToString("F4"))));
}

null


<null>

In [11]:
{
    // Exercice 3 — Une seule racine dans une fenêtre donnée ?
    bool? EstUnique(double sigma, double? kMax = null)
    {
        // TODO : compléter
        return null;  // TODO etudiant
    }

    var r3 = EstUnique(0.05);
    Console.WriteLine(r3 is null ? "null" : r3.Value.ToString());
}

null


<null>

### Synthèse — ce que les exercices verrouillent

Chaque stub vise **un** des faits mesurés ci-dessus, pas le carnet en général :

- **Exercice 1** (`ProbBothL`) — la sélection robuste **côté L** : symétrique de `ProbBothH`, il doit rendre des probabilités tendant vers `1` quand `σ → 0` et `θ < 0` (l'équilibre (L,L), **efficace** dans ce cas, devient quasi certain).
- **Exercice 2** (`SeuilsEnFonctionDeSigma`) — la convergence `k* → C` : relire la colonne « plus petit k* » du balayage et vérifier que le seuil robuste se fige sur `C = 1`, **pas** sur `0`.
- **Exercice 3** (`EstUnique`) — la **non**-unicité : `CompteRacines` rend **2** sur la fenêtre mesurée, donc `EstUnique` doit répondre `false` — le jeu global sélectionne le plus petit seuil **sans** que le jeu soit à une seule racine.

Les trois stubs s'appuient sur les fonctions déjà mesurées dans ce carnet ; il n'y a rien à réinventer, seulement à **réassembler** ce que le carnet a établi.

## Références

- **Carlsson, H., & van Damme, E. (1993).** « Global Games and Equilibrium Selection », *Econometrica* 61(5), 989–1018. — introduction du jeu global et de la sélection d'équilibre sous information bruitée.
- **Morris, S., & Shin, H. S. (1998).** « Unique Equilibrium in a Model of Self-Fulfilling Currency Attacks », *American Economic Review* 88(3), 587–597. — application canonique aux attaques de change.

**Place dans la série** : `11e`, « Autour de 11 » — avec `11` (jeux bayésiens), `11b` (Lean), `11c` (trembles) et `11d` (raffinements séquentiels).